# muon_db SQL Sandbox

Use this notebook to play with the `muon_db` tables using SQL. It uses DuckDB over the Spark-written Parquet folders, so queries start quickly and do not require a Spark session.

The registered views are:

- Bronze: `bronze_event`, `bronze_muon`, `bronze_jet`, `bronze_met`, `bronze_trigger`
- Silver: `silver_event`, `silver_muon`, `silver_jet`, `silver_met`, `silver_trigger`
- Gold: `event_summary`, `dimuon`, `jet`, `met`

In [ ]:
from pathlib import Path
import sys

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
sys.path.insert(0, str(WORKSPACE))

from src.access.muon_db import connect_with_tables

connection, tables = connect_with_tables(WORKSPACE / "data" / "muon_db")
[table.name for table in tables]

## Helper

Edit the SQL string in any cell and call `sql(...)` to return a pandas DataFrame.

In [ ]:
def sql(query: str):
    return connection.execute(query).fetchdf()

## Catalog And Row Counts

In [ ]:
sql("""
SELECT 'bronze_event' AS table_name, count(*) AS rows FROM bronze_event
UNION ALL SELECT 'bronze_muon', count(*) FROM bronze_muon
UNION ALL SELECT 'bronze_jet', count(*) FROM bronze_jet
UNION ALL SELECT 'bronze_met', count(*) FROM bronze_met
UNION ALL SELECT 'bronze_trigger', count(*) FROM bronze_trigger
UNION ALL SELECT 'silver_event', count(*) FROM silver_event
UNION ALL SELECT 'silver_muon', count(*) FROM silver_muon
UNION ALL SELECT 'silver_jet', count(*) FROM silver_jet
UNION ALL SELECT 'silver_met', count(*) FROM silver_met
UNION ALL SELECT 'silver_trigger', count(*) FROM silver_trigger
UNION ALL SELECT 'event_summary', count(*) FROM event_summary
UNION ALL SELECT 'dimuon', count(*) FROM dimuon
UNION ALL SELECT 'jet', count(*) FROM jet
UNION ALL SELECT 'met', count(*) FROM met
""")

## Event Summary Exploration

`event_summary` is the main event-level analytical table. It contains one row per cleaned event and includes counts, leading-object pT, MET, HT, and ST.

In [ ]:
sql("""
SELECT
    n_muons,
    n_jets,
    count(*) AS events,
    avg(MET_pt) AS avg_met,
    avg(HT) AS avg_ht,
    avg(ST) AS avg_st
FROM event_summary
GROUP BY n_muons, n_jets
ORDER BY events DESC
LIMIT 20
""")

## Cleaned Muon Exploration

`silver_muon` contains only muons that passed the cleaning rules: tight ID, isolation below 0.15, and pT above 20 GeV.

In [ ]:
sql("""
SELECT
    count(*) AS muons,
    min(pt) AS min_pt,
    avg(pt) AS avg_pt,
    max(pt) AS max_pt,
    avg(isolation) AS avg_isolation
FROM silver_muon
""")

## Dimuon Exploration

`dimuon` contains opposite-sign cleaned muon pairs. This is useful for resonance studies such as the Z peak.

In [ ]:
sql("""
SELECT
    count(*) AS pair_count,
    avg(invariant_mass) AS avg_mass,
    min(invariant_mass) AS min_mass,
    max(invariant_mass) AS max_mass
FROM dimuon
WHERE invariant_mass BETWEEN 70 AND 110
""")

## Custom Query Cell

Edit this query while exploring. All registered table names are available directly in SQL.

In [ ]:
custom_query = """
SELECT
    floor(MET_pt / 25) * 25 AS met_bin_low,
    count(*) AS events,
    avg(n_jets) AS avg_jets,
    avg(ST) AS avg_st
FROM event_summary
WHERE MET_pt IS NOT NULL
GROUP BY met_bin_low
ORDER BY met_bin_low
"""

sql(custom_query)